# Trabalho Fase 1 Turma 9IADT - Problemas Cardíacos - Grupo 86

A base de dados a ser análisada neste trabalho é sobre ataques de coração, onde foram colidos dados de 11 clinicas em diferentes países. (link da base de dados: https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction?resource=download)

## Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, roc_auc_score, classification_report, confusion_matrix

## Carregando os dados e analisando as informações

A base de dados que será análisada tem 12 colunas, onde se divide em:



1.   Age (Idade):
      Simplesmente a idade do paciente. É uma das variáveis mais importantes segundo dados médicos porque o risco de doenças cardíacas aumenta progressivamente com a idade. Isso acontece porque com o tempo as artérias perdem elasticidade, acumula-se placas de gordura nas paredes dos vasos e o coração vai sofrendo desgaste natural. ([A idade é realmente um fator crucial para doenças cardíacas?](https://www.drfernandofigueira.com.br/a-idade-e-realmente-um-fator-crucial-para-doencas-cardiacas))

2.   Sex (Sexo biológico):
      O sexo biológico do paciente. Homens têm risco cardiovascular significativamente maior do que mulheres antes da menopausa. Isso porque o estrogênio (hormônio feminino) tem efeito protetor sobre as artérias. Após a menopausa, o risco das mulheres sobe bastante e se aproxima ao dos homens. [Risco cardiovascular em homens aumenta a partir dos 35 anos, diz estudo](https://www.apm.org.br/risco-cardiovascular-em-homens-aumenta-a-partir-dos-35-anos-diz-estudo/)

3.  ChestPainType (Tipo de dor no peito):
      *   ASY — Assintomático:
            Sem dor no peito. É o tipo mais perigoso — a doença avança silenciosamente. Muito associado a HeartDisease = 1.
      *   ATA — Angina atípica:
            Dor que não segue o padrão clássico — pode ser queimação, pontada ou dor em outro local. Menos específica.
      *   TA — Angina típica:
            Dor clássica: aperto no peito que piora com esforço e melhora com repouso. Fortemente ligada ao coração.
      *   NAP — Dor não anginosa:
            Dor que provavelmente não vem do coração — pode ser muscular, digestiva, etc. Menor associação cardíaca.

4.  RestingBP (Pressão arterial em repouso):
      A pressão que o sangue faz nas paredes das artérias quando o coração está em repouso (entre batimentos). Pense numa mangueira: se a pressão da água for muito alta por muito tempo, as paredes se desgastam e podem romper. Com as artérias é igual — pressão cronicamente alta danifica as paredes e favorece a formação de placas. [Tabela de pressão arterial](https://telemedicinamorsch.com.br/blog/tabela-de-pressao-arterial?srsltid=AfmBOoqR2Wj2BPm7WE_lrKVxuv472M4MRQ5_8Q1glUovwumrx-TBkzEb)

5.  Cholesterol (Colesterol sérico):
      Quantidade de colesterol total circulando no sangue. O colesterol é uma gordura necessária pro organismo, mas em excesso se acumula nas paredes das artérias formando placas — chamadas de aterosclerose. Essas placas estreitam as artérias e podem causar infarto quando se rompem. [Quando o colesterol é considerado alto?](https://www.einstein.br/n/vida-saudavel/quando-colesterol-considerado-alto-saiba-o-que-influencua)

6.  FastingBS (Glicemia em jejum):
      Mede o nível de açúcar no sangue após pelo menos 8 horas de jejum. Já vem transformado em variável binária — não temos o valor exato, só saber se o paciente tem glicemia elevada (possível diabetes ou pré-diabetes) ou não. Diabetes é um fator de risco cardíaco porque o excesso de glicose danifica os vasos sanguíneos ao longo do tempo. [Glicemia em jejum](https://nav.dasa.com.br/blog/glicemia-em-jejum)

7.  RestingECG (Eletrocardiograma em repouso):
      O ECG (eletrocardiograma) registra a atividade elétrica do coração. É como "fotografar" os pulsos elétricos que fazem o coração bater. Quando o coração está saudável, esse gráfico tem um formato específico. Qualquer desvio pode indicar problemas. [Eletrocardiograma de Repouso](https://clinicabiocor.com.br/exame/eletrocardiograma-de-repouso/)

      *   Normal:
            Ritmo e padrão elétrico dentro do esperado. Sem sinais de problemas estruturais.
      *   ST — Alteração ST-T:
            Há uma parte do ECG chamada "segmento ST" que fica alterada. Pode indicar que parte do músculo cardíaco está recebendo menos sangue do que deveria (isquemia).
      *   LVH — Hipertrofia ventricular esquerda:
            O ventrículo esquerdo (câmara principal do coração) está maior que o normal — geralmente por anos de pressão alta. O músculo fica "exausto" de trabalhar contra tanta resistência.

8.  MaxHR (Frequência cardíaca máxima):
      A maior velocidade que o coração do paciente atingiu durante um teste de esforço (esteira ou bicicleta). É como medir o "rendimento máximo do motor" — um coração saudável consegue acelerar bastante durante exercício. Coração doente tem dificuldade de atingir frequências altas porque o fluxo de sangue já está comprometido.
      A FC máxima esperada para uma pessoa saudável é aproximadamente: 220 − idade. Então um paciente de 50 anos deveria atingir ~170 bpm num esforço máximo. [Qual é a frequência cardíaca máximar](https://www.tuasaude.com/frequencia-cardiaca-maxima/)

      *   Bom sinal:
            MaxHR alto → coração consegue trabalhar forte → menor risco
      *   Sinal de alerta:
            MaxHR baixo → coração não aguenta esforço → maior risco

9.  ExerciseAngina (Angina induzida por exercício):
      Indica se o paciente sentiu dor no peito (angina) durante o teste de esforço. Em repouso, o coração pode funcionar "bem o suficiente" mesmo com artérias parcialmente obstruídas. Mas quando o esforço exige mais sangue, a obstrução fica evidente e causa dor — é o equivalente a um carro que funciona na cidade mas para na estrada. [Dor ou aperto no peito durante o treino](https://cardiologistaemfoz.com.br/angina-no-exercicio-sinais-de-alerta-e-como-prevenir-crises/)

      *   N — Sem angina:
            O coração conseguiu suprir a demanda do esforço sem dor. Bom sinal.
      *   Y — Com angina:
            O coração não conseguiu suprir o esforço — forte indicador de obstrução coronária.

10.  Oldpeak (Depressão do segmento ST):
      Durante o esforço físico, o ECG pode mostrar que uma parte da curva (chamada "segmento ST") fica mais baixa do que deveria — esse rebaixamento se chama depressão do ST. Quanto mais fundo esse segmento cai, maior a evidência de que alguma região do músculo cardíaco está sofrendo falta de sangue durante o esforço. [O que é depressão ST](https://www.medicalnewstoday.com/articles/st-depression-on-ecg)

11.  ST_Slope (Inclinação do segmento ST):
      Ainda olhando para o segmento ST do ECG (durante esforço máximo), agora avaliamos a direção — se ele está subindo, plano ou descendo. Pense como a inclinação de uma linha: subindo é normal, plana ou descendo é sinal de problema. O coração saudável "responde bem" ao esforço, mantendo o ST ascendente. [Infradesnivelamento do segmento ST: causas, tipos e diagnóstico](https://telemedicinamorsch.com.br/blog/infradesnivelamento-do-segmento-st?srsltid=AfmBOoqD5J08GtPx2VJ-pWvfJwNwO01BfJYqllUi0LyChnbav8Unezm6)

      *   Up (ascendente) — Bom sinal:
            O coração responde normalmente ao esforço. A corrente elétrica sobe como esperado. Menor associação com doença.
      *   Flat (plano) — Atenção:
            ST fica horizontal — o coração não está respondendo bem ao esforço. É um sinal intermediário, muito associado a doenças cardíacas neste dataset.
      *   Down (descendente) — Alerta:
            ST desce durante o esforço. Indicador claro de isquemia — músculo cardíaco com falta de sangue.

12.  HeartDisease — (Diagnóstico final):
      Esta é a variável alvo — o que queremos prever. Indica se o paciente foi diagnosticado com alguma doença cardíaca (principalmente doença arterial coronariana — quando as artérias que alimentam o coração ficam obstruídas). É determinado pela combinação de todos os outros exames e sintomas.

      *   0 — Normal:
            Sem evidências de doença cardíaca significativa.
      *   Doença cardíaca:
            Diagnóstico confirmado de doença cardíaca.

In [ ]:
tabela = pd.read_csv("heart.csv")

In [ ]:
display(tabela.head())
print('\n')

tabela.info()
print('\n')

display(tabela.describe())
print('\n')

display(tabela.isnull().sum())
print('\n')

(tabela == 0).sum()
print('\n')

tabela.duplicated().sum()

Na análise acima identificamos que a base de dados contém o total de 12 colunas e 918 linhas, a váriavel escolhida como target será a HeartDisease. Na análise não foram encontrados quaisquer valores nulo, porem foram identificados valores zero em algumas colunas. Entretanto, zeros em HeartDisease, FastingBS e Oldpeak possuem significado válido. Já em Cholesterol e RestingBP, os valores zero são clinicamente inconsistentes e serão tratados no pré-processamento.

Tipos de colunas:
- Numéricas:
  - Age, RestingBP, Cholesterol, MaxHR, Oldpeak

 - Categóricas:
    - Sex, ChestPainType, RestingECG, ExerciseAngina, ST_Slope

 - Binárias:
    - FastingBS, HeartDisease

###Verificando a distribuição das variáveis


In [ ]:
print('Balanceamento de diagnóstico (HeartDisease)')
display(tabela["HeartDisease"].value_counts())

print('\nPorcentagem do balanceamento dos diagnósticos')
display(tabela["HeartDisease"].value_counts(normalize=True) * 100)

Observando os resultados acima, foi contatado que a base de dados escolhida está balanceada, onde temos uma proporção parecida de diagnósticos positivos e negativos

In [ ]:
print('Balanceamento de Sexo biológico (Sex)')
display(tabela["Sex"].value_counts())

print('\nPorcentagem do balanceamento do Sexo biológico')
display(tabela["Sex"].value_counts(normalize=True) * 100)

Para o Sexo Biológico temos um caso de tabela enviesado, onde temos uma proporção consideravelmente maior de homens que de mulheres

In [ ]:
print('Balanceamento de Tipo de dor no peito (ChestPainType)')
display(tabela["ChestPainType"].value_counts())

print('\nPorcentagem do balanceamento de Tipo de dor no peito')
display(tabela["ChestPainType"].value_counts(normalize=True) * 100)

Neste caso do tipo de dor no peito temos um valor exponencialmente maior para dores do tipo ASY, e bem poucas do tipo TA ficando em uma média as outras dores do tipo NAP e ATA

In [ ]:
print('Balanceamento de Glicemia em jejum (FastingBS)')
display(tabela["FastingBS"].value_counts())

print('\nPorcentagem do balanceamento de Glicemia em jejum')
display(tabela["FastingBS"].value_counts(normalize=True) * 100)

Para a glicemia em jejum a grande maioria (76.68% dos dados) não trazem alterações neste exame

In [ ]:
print('Balanceamento de Eletrocardiograma em repouso (RestingECG)')
display(tabela["RestingECG"].value_counts())

print('\nPorcentagem do balanceamento de Eletrocardiograma em repouso')
display(tabela["RestingECG"].value_counts(normalize=True) * 100)


display(tabela.head())

Para o Eletrocardiograma em repouso um pouco mais da metade dos dados estão em normal, ficando balanceado entre LVH e ST que são maiores pontos de atenção para o diagnóstico

In [ ]:
print('Balanceamento de Angina induzida por exercício (ExerciseAngina)')
display(tabela["ExerciseAngina"].value_counts())

print('\nPorcentagem do balanceamento de Angina induzida por exercícioo')
display(tabela["ExerciseAngina"].value_counts(normalize=True) * 100)

Para a Angina induzida por exercício a base de dados está relativamente balanceada, onde temos 59.58% que não tiveram angina induzida por exercício e 40.41% que tiveram

In [ ]:
print('Balanceamento de Inclinação do segmento ST (ST_Slope)')
display(tabela["ST_Slope"].value_counts())

print('\nPorcentagem do balanceamento de Inclinação do segmento ST')
display(tabela["ST_Slope"].value_counts(normalize=True) * 100)

Em quesito a inclinação do segmento ST, temos uma base balanceada entre Flat e Up e pouca representatividade sobre a variável Down

### Analíse em gráficos das variáveis

Verificando da distribuição da coluna Age (idade) com Heartdisease

In [ ]:
sns.histplot(data=tabela, x="Age", hue="HeartDisease", bins=20, kde=True)
plt.title("Idade por presença de doença cardíaca")
plt.show()

print('\n')
tabela.groupby("HeartDisease")["Age"].describe()

A análise da distribuição etária revela que o risco de doenças cardíacas tende a aumentar com a idade. Observamos uma concentração maior de casos positivos a partir dos 50 anos, com a mediana de idade para pacientes diagnosticados sendo superior à dos pacientes saudáveis. Isso segue a literatura médica sobre o desgaste cardiovascular ao longo do tempo.

In [ ]:
sns.countplot(data=tabela, x="Sex", hue="HeartDisease")
plt.title("Doença cardíaca por sexo")
plt.show()

print('\n')
pd.crosstab(
    tabela["Sex"],
    tabela["HeartDisease"],
    normalize="index"
) * 100

Nota-se uma disparidade significativa entre os sexos: enquanto a maioria das mulheres na amostra não apresenta a doença, mais de 60% dos homens foram diagnosticados positivamente. Este dado é um forte indicativo de que o sexo biológico masculino é um fator de risco relevante nesta base de dados.

In [ ]:
sns.histplot(data=tabela[tabela["Cholesterol"] > 0], x="Cholesterol", hue="HeartDisease", bins=30, kde=True)
plt.title("Distribuição do colesterol sem valores zero")
plt.show()

print('\n')
tabela[tabela["Cholesterol"] > 0].groupby("HeartDisease")["Cholesterol"].describe()

Embora a média de colesterol seja ligeiramente superior no grupo com doença cardíaca, as distribuições são sobrepostas, indicando que o colesterol isoladamente pode não ser o único fator determinante, mas sim um componente que atua em conjunto com outros indicadores metabólicos.

In [ ]:
sns.histplot(
    data=tabela[tabela["RestingBP"] > 0],
    x="RestingBP",
    hue="HeartDisease",
    bins=30,
    kde=True
)

plt.title("Distribuição da pressão arterial em repouso sem valores zero")
plt.show()

print('\n')
tabela[tabela["RestingBP"] > 0].groupby("HeartDisease")["RestingBP"].describe()

A pressão arterial em repouso apresenta uma distribuição muito similar entre os grupos. Isso sugere que, embora a hipertensão seja um risco conhecido, a medição pontual em repouso nesta base de dados possui um poder discriminatório menor do que variáveis dinâmicas (como as medidas durante esforço).

In [ ]:
sns.histplot(
    data=tabela,
    x="MaxHR",
    hue="HeartDisease",
    bins=30,
    kde=True
)

plt.title("MaxHR por presença de doença cardíaca")
plt.show()

print('\n')
tabela.groupby("HeartDisease")["MaxHR"].describe()

A Frequência Cardíaca Máxima (MaxHR) mostra-se um excelente indicador: pacientes com doença cardíaca tendem a atingir picos de frequência menores durante o esforço. A separação visual no histograma é nítida, sugerindo que a capacidade de resposta do coração ao estresse físico é um dos principais diferenciais clínicos.

In [ ]:
sns.histplot(
    data=tabela,
    x="Oldpeak",
    hue="HeartDisease",
    bins=30,
    kde=True
)

plt.title("Oldpeak por presença de doença cardíaca")
plt.show()

print('\n')
tabela.groupby("HeartDisease")["Oldpeak"].describe()

O Oldpeak (depressão do segmento ST) é um dos indicadores mais fortes de anomalia. Valores próximos de zero são predominantes em pacientes saudáveis, enquanto valores mais altos estão massivamente associados à presença de doença cardíaca, evidenciando sofrimento do músculo cardíaco sob esforço.

In [ ]:
sns.countplot(data=tabela, x="ChestPainType", hue="HeartDisease")
plt.title("Tipo de dor no peito por HeartDisease")
plt.show()

print('\n')
pd.crosstab(
    tabela["ChestPainType"],
    tabela["HeartDisease"],
    normalize="index"
) * 100

Surpreendentemente, o tipo de dor 'Assintomático' (ASY) é o que possui a maior taxa de diagnósticos positivos (cerca de 79%). Isso alerta para a periculosidade de doenças silenciosas, onde a ausência de dor típica não exclui a presença de patologias graves.

In [ ]:
sns.countplot(data=tabela, x="ExerciseAngina", hue="HeartDisease")
plt.title("Angina induzida por exercício por HeartDisease")
plt.show()

print('\n')
pd.crosstab(
    tabela["ExerciseAngina"],
    tabela["HeartDisease"],
    normalize="index"
) * 100

A presença de angina induzida por exercício (ExerciseAngina = Y) é um preditor clínico muito forte: mais de 85% dos pacientes que sentiram dor durante o esforço tiveram a doença confirmada, validando a importância dos testes de estresse para diagnóstico.

In [ ]:
sns.countplot(data=tabela, x="ST_Slope", hue="HeartDisease")
plt.title("ST_Slope por HeartDisease")
plt.show()

print('\n')
pd.crosstab(
    tabela["ST_Slope"],
    tabela["HeartDisease"],
    normalize="index"
) * 100

O comportamento do segmento ST no pico do esforço (ST_Slope) é crucial. Enquanto a inclinação ascendente (Up) está associada à saúde, as inclinações planas (Flat) e descendentes (Down) dominam os casos de doença, refletindo falhas na repolarização elétrica do coração.

In [ ]:
corr = tabela.corr(numeric_only=True)
corr["HeartDisease"].sort_values(ascending=False)

A análise de correlação de Pearson reforça as observações visuais feitas anteriormente, destacando os seguintes pontos:

1.  **Oldpeak (+0.40):** Apresenta a maior correlação positiva, indicando que quanto maior a depressão do segmento ST (sinal de estresse cardíaco), maior a probabilidade de presença de doença.
2.  **MaxHR (-0.40):** Apresenta uma correlação negativa significativa. Isso sugere que a incapacidade do coração de atingir frequências mais altas durante o esforço é um forte indicador clínico de comprometimento cardiovascular.
3.  **Age (+0.28) e FastingBS (+0.27):** Embora menores, indicam que a idade avançada e níveis elevados de glicemia em jejum são fatores que caminham proporcionalmente ao risco de diagnóstico positivo.
4.  **Cholesterol (-0.23):** Curiosamente, apresenta uma correlação negativa moderada. Isso pode ocorrer devido ao tratamento medicamentoso de pacientes já diagnosticados ou à presença de valores discrepantes, reforçando que o colesterol deve ser analisado dentro de um contexto multifatorial e não isoladamente neste conjunto de dados.

## Etapa de Pré-processamento

In [ ]:
df = tabela.copy()
# Substituindo zeros inválidos da coluna Cholesterol e da coluna RestingBP pelas medianas
df["Cholesterol"] = df["Cholesterol"].replace(0, pd.NA)
df["RestingBP"] = df["RestingBP"].replace(0, pd.NA)

df["Cholesterol"] = df["Cholesterol"].fillna(df["Cholesterol"].median())
df["RestingBP"] = df["RestingBP"].fillna(df["RestingBP"].median())

 Foi feita uma cópia da tabela original para trabalhar os dados sem alterar diretamente a base inicial. Depois, os valores iguais a zero nas colunas Cholesterol e RestingBP foram substituídos por NA, esses zeros não representam valores reais para essas variáveis, já que colesterol e pressão arterial em repouso não deveriam ser iguais a zero. Em seguida, os valores ausentes foram preenchidos com a mediana de cada coluna. A mediana foi escolhida por ser uma medida mais resistente a valores extremos, ajudando a manter os dados mais consistentes para a análise e para o treinamento do modelo.

In [ ]:
# Separação entre variável explicativa e Target
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

Aqui foi realizado a separação das variáveis explicativas armazenadas em X, dropando a target e no y apenas a coluna da taget foi armazenada (HeartDisease)

Etapa de Pipeline de pré-processamento

In [ ]:
# Definindo colunas categóricas e colunas numéricas
colunas_categoricas = ["Sex", "ChestPainType", "RestingECG", "ExerciseAngina", "ST_Slope"]
colunas_numericas = ["Age", "RestingBP", "Cholesterol", "FastingBS", "MaxHR", "Oldpeak"]

# Fazendo o tratamento das colunas numéricas e das categóricas
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), colunas_numericas),
        ("cat", OneHotEncoder(handle_unknown="ignore"), colunas_categoricas)
    ]
)

Nessa etapa, foram definidas separadamente as colunas categóricas e numéricas da base de dados. As colunas numéricas receberam o tratamento com **StandardScaler**, que padroniza os valores para deixá-los em uma escala parecida, evitando que variáveis com números maiores influenciem mais o modelo. Já as colunas categóricas receberam o tratamento com **OneHotEncoder**, que transforma categorias em colunas numéricas para que possam ser utilizadas pelo algoritmo. Para aplicar esses dois tratamentos ao mesmo tempo, foi utilizado o **ColumnTransformer**, que permite definir transformações específicas para cada grupo de colunas. O parâmetro handle_unknown="ignore" foi usado para evitar erros caso surjam categorias novas durante a etapa de teste.

In [ ]:
# Aplicando o pré-processamento para obter os dados transformados
X_transformed = preprocessor.fit_transform(X)

# Obtendo os nomes das colunas após o OneHotEncoder
nomes_colunas_cat = preprocessor.named_transformers_['cat'].get_feature_names_out(colunas_categoricas)
nomes_colunas_totais = list(colunas_numericas) + list(nomes_colunas_cat)

# Criando um novo DataFrame com os dados tratados e incluindo a Target
df_processado = pd.DataFrame(X_transformed, columns=nomes_colunas_totais)
df_processado['HeartDisease'] = y.values

# Calculando a correlação
correlation_matrix = df_processado.corr().round(2)

# Plotando o Heatmap incluindo a target
plt.figure(figsize=(18, 14))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlação com Variáveis Tratadas e Target')
plt.show()

A visualização da matriz de correlação após o pré-processamento  nos permite observar métricas que não eram viziveis nas análises iniciais

1.  **ST_Slope (+0.55 e -0.62):** O valor Up apresenta uma forte correlação negativa com a doença cardíaca (-0.62), confirmando que uma é um forte indicador de saúde. Por outro lado, Flat (+0.55) é o indicador categórico mais fortemente associado à presença da doença.
2.  **ChestPainType (+0.52):** Entre os tipos de dor no peito, a condição assintomática (ASY) é a que mais se correlaciona positivamente com o diagnóstico, reforçando que a ausência de dor típica é um sinal de alerta crítico neste dataset.
3.  **ExerciseAngina (+0.49):** A presença de angina induzida por exercício mostra uma correlação positiva robusta, sendo um dos preditores clínicos mais confiáveis.
4.  **Sex (+0.31 e -0.31):** A correlação confirma que ser do sexo masculino aumenta a probabilidade estatística de diagnóstico positivo nesta amostra, enquanto ser do sexo feminino atua como um fator inversamente correlacionado.
5.  **Outras variáveis:** Oldpeak e MaxHR mantêm sua relevância, mostrando que o sofrimento elétrico do coração e a baixa frequência cardíaca são pilares para a identificação da patologia.

## Separando as bases de treino e teste

A base de dados foi separada em 80% para treino e 20% para teste, utilizando o valor padrão de random_state de 42 e especificado qual variável é a target ( y = HeartDisease )

In [ ]:
# Separando treino e teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Escolhendo o melhor modelo para ser utilizado

Para a previsão de doenças cardíacas, foram utilizados três modelos distintos:

* **Regressão Logística:** Foi utilizado por ser um modelo simples, eficiente e adequado para problemas de classificação binária. Ela foi importante como modelo inicial de comparação, permitindo avaliar se uma abordagem menos complexa já seria capaz de apresentar bom desempenho na previsão da presença de doença cardíaca.
* **Random Forest:** O Random Forest foi utilizado por ser um modelo mais robusto que a Regressão Logística, capaz de capturar relações mais complexas entre as variáveis. Como ele combina várias árvores de decisão, tende a reduzir o risco de overfitting em comparação com uma árvore isolada e pode apresentar boa capacidade de generalização.
* **Gradient Boosting:** Escolhido pela sua alta performance em dados estruturados, trabalhando de forma iterativa para minimizar os erros dos classificadores anteriores.

Para os modelos descritos acima, foram obtidas as seguintes métricas:

| Modelo | Accuracy | Precision | Recall | F1-Score | ROC-AUC |
| :--- | :---: | :---: | :---: | :---: | :---: |
| **Logistic Regression** | 0.8858 | 0.8857 | 0.9117 | 0.8985 | 0.9330 |
| **Random Forest** | 0.8804 | 0.8773 | 0.9117 | 0.8942 | 0.9233 |
| **Gradient Boosting** | 0.8804 | **0.8921** | 0.8921 | 0.8921 | 0.9183 |

In [ ]:
# Importando os modelos que serão testados e as métricas de avaliação


# Aqui vamos testar três modelos diferentes para comparar qual deles se sai melhor
modelos = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# Lista onde serão guardados os resultados de cada modelo
resultados = []

# Treinando e avaliando cada modelo individualmente
for nome, classificador in modelos.items():

    # Criando um pipeline que primeiro aplica o pré-processamento
    # e depois treina o modelo de classificação
    modelo = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", classificador)
    ])

    # Treinando o modelo com os dados de treino
    modelo.fit(X_train, y_train)

    # Gerando as previsões com os dados de teste
    y_pred = modelo.predict(X_test)

    # Gerando as probabilidades da classe positiva, ou seja, presença de doença cardíaca
    y_proba = modelo.predict_proba(X_test)[:, 1]

    # Salvando as principais métricas de avaliação do modelo
    resultados.append({
        "Modelo": nome,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_proba)
    })

In [ ]:
# Transformando os resultados em um DataFrame para facilitar a comparação
resultados_df = pd.DataFrame(resultados)

# Ordenando os modelos pelo Recall, pois nesse problema é importante identificar corretamente os pacientes com doença
resultados_df.sort_values(by="Recall", ascending=False)

## Treinando e analisando o melhor modelo encontrado

Com base nos resultados, o modelo selecionado para este caso foi o **Gradient Boosting**. 

Embora os modelos tenham apresentado desempenhos muito próximos, o **Gradient Boosting** destacou-se por apresentar a maior **Precisão (0.8921)**. Como se trata de um contexto médico, optou-se por utilizar essa métrica como a principal. Após essa escolha, foi realizado o treinamento do modelo e foi utilizado o classification report como forma de analisar as métricas de resultado, obtendo os seguintes valores:

*   Classificados como diagnóstico negativo:

    *   precision: 0.87,
    *   Recall: 0.87,
    *   F1-score: 0.87;


*   Classificados como diagnóstico positivo:

    *   precision: 0.89,
    *   Recall: 0.89,
    *   F1-score: 0.89;

In [ ]:
# Importando ferramentas para visualizar melhor o desempenho do modelo escolhido


# Criando novamente o modelo escolhido para uma análise mais detalhada
# Neste exemplo foi usado Gradient Boosting, mas você pode trocar pelo modelo com melhor resultado na tabela anterior
melhor_modelo = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", GradientBoostingClassifier(random_state=42))
])

# Treinando o modelo escolhido
melhor_modelo.fit(X_train, y_train)

# Fazendo previsões com o conjunto de teste
y_pred = melhor_modelo.predict(X_test)

# Exibindo o relatório com Precision, Recall, F1-score e Accuracy
print(classification_report(y_test, y_pred))

Como uma última análise, foi utilizada uma matriz e confusão para que fosse possível analisar a quantidade de falsos positivos e falsos negativos que o modelo previu, obtendo as seguintes métricas:

*   De **82** casos onde o modelo fez a previsão de diagnóstico negativo, **71** deles foram corretos e **11** foram falsos positivos;
*   De **102** casos onde o modelo previu como diagnóstico positivo **91** deles foram corretos e **11** foram falso negativos.

In [ ]:
# Criando a matriz de confusão para visualizar acertos e erros do modelo
matriz = confusion_matrix(y_test, y_pred)

# Plotando a matriz de confusão
sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues")

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.title("Matriz de Confusão - Melhor Modelo")
plt.show()

## Conclusão

A análise dos dados e o desenvolvimento dos modelos preditivos revelam que é possível identificar padrões críticos associados a doenças cardíacas com alta precisão. Variáveis como a resposta do segmento ST durante o esforço (ST_Slope), a presença de angina induzida e a frequência cardíaca máxima mostraram-se indicadores fundamentais para a triagem de pacientes.

Contudo, é importante ressaltar que o modelo atual apresenta limitações para uso clínico imediato. O desbalanceamento na representatividade entre homens e mulheres na base de dados original pode introduzir tendências que não refletem com exatidão a realidade de todos os perfis de pacientes. Na prática médica, decisões baseadas em algoritmos devem sempre considerar a diversidade da amostra para evitar diagnósticos imprecisos.

Em suma, o modelo desenvolvido pode atuar como uma ferramenta auxiliar de triagem eficiente, ajudando profissionais de saúde a priorizar casos de alto risco e a aprofundar investigações em quadros assintomáticos que, sem esse suporte tecnológico, poderiam passar despercebidos.